# Exploracao do dataset arXiv

Notebook para carregar o JSONL mais recente em `data/bronze/arxiv/raw`, inspecionar o schema e praticar tecnicas de manipulacao de dados com `pandas`.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
plt.style.use("ggplot")


In [ ]:
RAW_DIR = Path("../data/bronze/arxiv/raw")
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Diretorio nao encontrado: {RAW_DIR.resolve()}")

candidate_files = sorted(RAW_DIR.glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
if not candidate_files:
    raise FileNotFoundError(f"Nenhum arquivo JSONL encontrado em {RAW_DIR.resolve()}")

dataset_path = candidate_files[-1]
dataset_path


In [ ]:
records = [
    json.loads(line)
    for line in dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.DataFrame(records)
df["published_at"] = pd.to_datetime(df["published"], utc=True, errors="coerce")
df["updated_at"] = pd.to_datetime(df["updated"], utc=True, errors="coerce")
df["title_len"] = df["title"].fillna("").str.len()
df["summary_len"] = df["summary"].fillna("").str.len()
df["authors_n"] = df["authors"].apply(lambda value: len(value) if isinstance(value, list) else 0)
df["categories_n"] = df["categories"].apply(lambda value: len(value) if isinstance(value, list) else 0)

print(f"Arquivo: {dataset_path}")
print(f"Linhas carregadas: {len(df):,}")
df.shape


In [ ]:
def safe_nunique(series: pd.Series) -> int:
    normalized = series.apply(
        lambda value: tuple(value) if isinstance(value, list) else value
    )
    return normalized.nunique(dropna=True)


profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.apply(safe_nunique),
    }
).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

profile


In [ ]:
df[
    [
        "arxiv_id",
        "title",
        "primary_category",
        "published_at",
        "authors_n",
        "categories_n",
        "summary_len",
    ]
].head(10)


## Explode de listas e agregacoes

As colunas `authors` e `categories` sao listas. A tecnica mais util aqui e `explode`, seguida de `groupby`, `value_counts` e joins com o dataframe original.


In [ ]:
categories_df = (
    df[["arxiv_id", "title", "published_at", "primary_category", "categories"]]
    .explode("categories")
    .rename(columns={"categories": "category"})
)

category_counts = categories_df["category"].value_counts().head(15)
ax = category_counts.plot(kind="bar", figsize=(12, 4), title="Top 15 categorias")
ax.set_xlabel("categoria")
ax.set_ylabel("artigos")
plt.xticks(rotation=45, ha="right")
plt.show()

category_counts


In [ ]:
authors_df = (
    df[["arxiv_id", "published_at", "authors"]]
    .explode("authors")
    .rename(columns={"authors": "author"})
)

authors_df["author"].value_counts().head(20)


In [ ]:
monthly_counts = (
    df.dropna(subset=["published_at"])
    .assign(month=lambda frame: frame["published_at"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp())
    .groupby("month")
    .size()
)

ax = monthly_counts.tail(24).plot(figsize=(12, 4), title="Artigos por mes (ultimos 24 meses)")
ax.set_xlabel("mes")
ax.set_ylabel("artigos")
plt.show()

monthly_counts.tail()


## Manipulacoes uteis para estudo

Este bloco combina `query`, `assign`, filtros por data, ordenacao e selecao de colunas para gerar uma visao mais enxuta do dataset.


In [ ]:
cutoff = pd.Timestamp("2025-01-01", tz="UTC")

recent_articles = (
    df.loc[df["published_at"] >= cutoff]
    .assign(
        is_multi_category=lambda frame: frame["categories_n"] > 1,
        has_doi=lambda frame: frame["doi"].notna(),
        first_author=lambda frame: frame["authors"].apply(
            lambda value: value[0] if isinstance(value, list) and value else None
        ),
    )
    .sort_values(["published_at", "summary_len"], ascending=[False, False])
)

recent_articles[
    [
        "published_at",
        "primary_category",
        "title",
        "first_author",
        "is_multi_category",
        "has_doi",
        "summary_len",
    ]
].head(20)


In [ ]:
OUTPUT_DIR = Path("../data/silver/exploration")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

curated = (
    df[
        [
            "arxiv_id",
            "title",
            "primary_category",
            "published_at",
            "updated_at",
            "authors_n",
            "categories_n",
            "summary_len",
            "doi",
            "pdf_url",
        ]
    ]
    .sort_values("published_at", ascending=False)
    .reset_index(drop=True)
)

csv_path = OUTPUT_DIR / "arxiv_curated_sample.csv"
curated.to_csv(csv_path, index=False)

try:
    parquet_path = OUTPUT_DIR / "arxiv_curated_sample.parquet"
    curated.to_parquet(parquet_path, index=False)
    parquet_status = str(parquet_path)
except Exception as exc:
    parquet_status = f"parquet nao gerado: {exc}"

{
    "csv": str(csv_path),
    "parquet": parquet_status,
    "rows": len(curated),
}


## Proximos passos

- Trocar o filtro de data e criar subconjuntos por categoria.
- Gerar features derivadas como ano, mes, primeiro autor e flags de DOI.
- Exportar tabelas normalizadas (`authors_df`, `categories_df`) para analise relacional.
- Testar joins entre o dataframe original e os dataframes explodidos para enriquecer consultas.
